Import Libraries and Set Seeds

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import random, numpy as np
from google.colab import drive

SEED = 42
## Set seeds for all random sources
random.seed(SEED)        # Python built-in RNG
np.random.seed(SEED)     # NumPy RNG
tf.random.set_seed(SEED) # TensorFlow RNG

Import Data

In [ ]:
#  Dataset paths
L1 = ''
L2 = ''
L3 = ''
L4 = ''

#  Load Training set
X_train = np.load(L1)
Y_train = np.load(L2)
# Load Dev set
X_val = np.load(L3)
Y_val = np.load(L4)

### Post Process --> remove Nans and Infinities (Some Vs30 Values may be Nan) --> Replace with site average
X_train_pre = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
Y_train_pre = np.nan_to_num(Y_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val_pre = np.nan_to_num(X_val,   nan=0.0, posinf=0.0, neginf=0.0)
Y_val_pre  = np.nan_to_num(Y_val,   nan=0.0, posinf=0.0, neginf=0.0)
#Post process X-train and Y-train
X_train = X_train_pre.copy()
channel_means = np.nanmean(X_train, axis=(1, 2), keepdims=True)
nan_mask = np.isnan(X_train)
X_train = np.where(nan_mask, channel_means, X_train)
Y_train = Y_train_pre.copy()
channel_means = np.nanmean(Y_train, axis=(1, 2), keepdims=True)
nan_mask = np.isnan(Y_train)
Y_train = np.where(nan_mask, channel_means, Y_train)
#Post process X-val and Y-val
X_val = X_val_pre.copy()
channel_means = np.nanmean(X_val, axis=(1, 2), keepdims=True)
nan_mask = np.isnan(X_val)
X_val = np.where(nan_mask, channel_means, X_val)
Y_val = Y_val_pre.copy()
channel_means = np.nanmean(Y_val, axis=(1, 2), keepdims=True)
nan_mask = np.isnan(Y_val)
Y_val = np.where(nan_mask, channel_means, Y_val)
# Check all Nans and infinities removed
print("X_train has NaN:", np.isnan(X_train).any(), "Inf:", np.isinf(X_train).any())
print("Y_train has NaN:", np.isnan(Y_train).any(), "Inf:", np.isinf(Y_train).any())
print("X_val has NaN:", np.isnan(X_val).any(), "Inf:", np.isinf(X_val).any())
print("Y_val has NaN:", np.isnan(Y_val).any(), "Inf:", np.isinf(Y_val).any())

#Check imported shapes
print(" ")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)

In [ ]:
#View some of the ground truth shakemaps before training
plt.figure(figsize=(4, 4))
for i in range(4):
    idx = i + 1000    # starting index
    plt.subplot(2, 2, i + 1)
    plt.title(f"Y_train #{idx}")
    plt.imshow(Y_train[idx])
    plt.axis("off")
plt.tight_layout()
plt.show()

Build Model

In [ ]:
#Input and Output Volume Dimensions
H_in, W_in, C_in = X_train.shape[1:4]
H_out, W_out, C_out = Y_train.shape[1:4]

# Model definition (6 layers)
X = layers.Input(shape=(H_in, W_in, C_in))

# Add input normalization layer
norm_layer = layers.Normalization(axis=-1)  #Normalize Across Channels --> Each channel holds different kind of data
norm_layer.adapt(X_train)  # Compute per-channel mean and variance across entire training set
X_norm = norm_layer(X)  # Normalize the input

#2 Conv layers to extra spatial information
a1 = layers.Conv2D(filters=16, kernel_size=7, strides=1, padding='same', activation='relu')(X_norm)
a2 = layers.Conv2D(filters=16, kernel_size=5, strides=1, padding='same', activation='relu')(a1)

#2 tranposed conv layers with cropping to grow spatial dimensions
a3 = layers.Conv2DTranspose(filters=16, kernel_size=3, strides=2, padding='valid', activation='relu')(a2)
a3 = layers.Cropping2D(cropping=((13, 13), (13, 13)))(a3)  #cropping replicates tconv padding of 13
a4 = layers.Conv2DTranspose(filters=8, kernel_size=6, strides=2, padding='valid', activation='relu')(a3)
a4 = layers.Cropping2D(cropping=((2, 2), (2, 2)))(a4)  # cropping replicates tconv padding of 2

#Conv layer before output layer
a5 = layers.Conv2D(filters=8, kernel_size=3, strides=1, padding='same', activation='relu')(a4)

#Output layer: 1x1 conv with sigmoid activation to produce [0,1] RGBs
y_hats = layers.Conv2D(filters=C_out, kernel_size=1, strides=1, padding='same', activation='sigmoid')(a5)

#Insantiate model
model = models.Model(inputs = X, outputs = y_hats, name = "Heatmap_Shakemap_Generator")

Compile for training

In [ ]:
# Compile
opt = tf.keras.optimizers.Adam(learning_rate=1e-3)   #Adam optimizer with alpha = 0.001
model.compile(optimizer = opt, loss = normalized_l2_RGB_loss)  #Global Pixel Error Loss

# Inspect model
model.summary()

Train Model and Cross Validate

In [ ]:
# Training
BATCH_SIZE = 16
EPOCHS = 25

HISTORY = model.fit(
    X_train, Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    validation_data=(X_val, Y_val)
)

View Loss Graphs

In [ ]:
# Plot both training and validation losses
plt.plot(HISTORY.history['loss'], label='Train')
plt.plot(HISTORY.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss vs Epochs')
plt.legend()
plt.show()

Inspect Some Dev Samples

In [ ]:
Y_pred_val = model.predict(X_val)
Y_pred_val = np.asarray(Y_pred_val)

plus = 12
for i in range(12):  # change number as desired
    plt.figure(figsize=(4,4))

    plt.subplot(1,2,1)
    plt.title(f"Ground Truth #{i+plus}")
    plt.imshow(Y_val[i+plus])
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title(f"Prediction #{i+plus}")
    plt.imshow(Y_pred_val[i+plus])
    plt.axis("off")

    plt.show()

Inspect Some Training Samples

In [ ]:
Y_pred_train = model.predict(X_train)
Y_pred_train = np.asarray(Y_pred_train)

for i in range(12):
    plt.figure(figsize=(4,4))

    plt.subplot(1,2,1)
    plt.title(f"Ground Truth (Train) #{i+60}")
    plt.imshow(Y_train[i+60])
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title(f"Prediction (Train) #{i+60}")
    plt.imshow(Y_pred_train[i+60])
    plt.axis("off")

    plt.show()

Evaluate Model via Performance Metrics


In [ ]:
# Training metrics
mad_train  = mad_eval(Y_train, Y_pred_train)
ssim_train = ssim_eval_batch(Y_train, Y_pred_train)

print("Training MAD :", mad_train)
print("Training SSIM:", ssim_train)

# Validation metrics
mad_val  = mad_eval(Y_val, Y_pred_val)
ssim_val = ssim_eval_batch(Y_val, Y_pred_val)

print("Validation MAD :", mad_val)
print("Validation SSIM:", ssim_val)

# Mean Score and STDV Score
# train
mad_mean_tr, mad_std_tr = mad_train
ssim_mean_tr, ssim_std_tr = ssim_train

mean_score_tr = 0.5 * (1 - mad_mean_tr) + 0.5 * ssim_mean_tr
std_score_tr  = 0.5 * mad_std_tr + 0.5 * ssim_std_tr

print("Training Mean Score:", mean_score_tr)
print("Training STDV Score:", std_score_tr)

# Validation
mad_mean_val, mad_std_val = mad_val
ssim_mean_val, ssim_std_val = ssim_val

mean_score_val = 0.5 * (1 - mad_mean_val) + 0.5 * ssim_mean_val
std_score_val  = 0.5 * mad_std_val + 0.5 * ssim_std_val

print("Validation Mean Score:", mean_score_val)
print("Validation STDV Score:", std_score_val)

Loss Functions

In [ ]:
###########   Pixel-wise RGB (flatten entire 250x250x3 image)  ###################
#|| y - y_hat ||_2  / ( || y ||_2 + eps )
@tf.function
def normalized_l2_RGB_loss(y_true, y_pred, eps=tf.constant(1e-8, dtype=tf.float32)):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    N = tf.shape(y_true)[0]    #number of samples in batch

    #Numerator --> Calculate element-wise difference (error) then flatten
    diff = tf.reshape(y_true - y_pred, [N, -1])
    numer = tf.sqrt(tf.reduce_sum(diff * diff, axis=1))    #L2 norm of difference
    #Denominator --> Normalization
    gtruth = tf.reshape(y_true, [N, -1])
    denom = tf.sqrt(tf.reduce_sum(gtruth * gtruth, axis=1)) + eps   #ground truth L2 norm

    #Calculate loss and batch cost
    per_example = numer / denom  #individual loss per example
    return tf.reduce_mean(per_example)  #Cost over batch


###########   Pixel-wise RGB (sum over channels)  ###################
#average [|| yc - yc_hat ||_2  / ( || yc ||_2 + eps )] over the three channels
@tf.function
def normalized_l2_RGB_channelwise_loss(y_true, y_pred, eps=tf.constant(1e-8, dtype=tf.float32)):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    #Difference between y_pred and y_hat pixels
    diff = y_true - y_pred                     # (B, H, W, 3)

    #Numerator --> average of channel-wise difference L2 norm
    numer = tf.sqrt(tf.reduce_sum(diff * diff, axis=[1, 2]))             # (B, 3)
    numer = tf.reduce_mean(numer, axis=1)

    #Denominator --> Normalization by global (denominator is not channel wise) ground truth L2 norm
    N = tf.shape(y_true)[0]    #number of samples in batch
    gtruth = tf.reshape(y_true, [N, -1])
    denom = tf.sqrt(tf.reduce_sum(gtruth * gtruth, axis=1)) + eps

    #Calculate loss and batch cost
    per_example = numer / denom  #individual loss per example
    return tf.reduce_mean(per_example)  #Cost over batch


########## SSIM Loss #################
def ssim_loss_tf(y_true, y_pred, max_val=1.0):
    """
    Computes SSIM loss using TensorFlow's built-in SSIM.

    y_true, y_pred: (B, H, W, C), pixel values expected in [0, max_val]
    Returns: scalar loss = 1 - mean(SSIM)
    """
    # Cast to float
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Built-in SSIM → (B,)
    ssim_vals = tf.image.ssim(y_true, y_pred, max_val=max_val)

    # Convert similarity to loss → scalar
    return 1.0 - tf.reduce_mean(ssim_vals)

########## Combined Loss ##########
"""
Combined channel-wise RGB pixel loss and SSIM loss. Weights of 0.75 and 0.25, respectively
"""
def combined_rgb_ssim_loss(y_true, y_pred, w_pix=0.75, w_ssim=0.25):
    L_pix  = normalized_l2_RGB_channelwise_loss(y_true, y_pred)
    L_ssim = ssim_loss_tf(y_true, y_pred)
    return w_pix * L_pix + w_ssim * L_ssim

Evluation Metric Functions

In [ ]:
########### Mean and STDV Absolute RGB Pixel Difference ###########
"""
Difference vector between flattened ground-truth vector and flattened prediction.
Then average absolute difference across the three channels for a single scalar value.
Average this scalar value from each of the training examples across the dataset,
and get STDV.
"""
def mad_eval(y_true, y_pred):
    per_img_mad = np.mean(np.abs(y_true - y_pred), axis=(1,2,3))
    M = float(np.mean(per_img_mad))   #Mean
    S = float(np.std(per_img_mad))  #STDV
    return M,S


########### Meand and STDV SSIM Across Dataset ###########
"""
Calculate SSIM between ground truth image and prediction. Then average SSIM's
across the dataset and get STDV.
"""
def ssim_eval(y_true, y_pred):
    ssim_vals = tf.image.ssim(tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32), max_val=1.0).numpy()
    M = float(np.mean(ssim_vals))
    S = float(np.std(ssim_vals))
    return M,S

def ssim_eval_batch(y_true, y_pred, batch_size=32):
    ssim_list = []
    N = y_true.shape[0]

    for i in range(0, N, batch_size):
        y_t = y_true[i:i+batch_size]
        y_p = y_pred[i:i+batch_size]

        vals = tf.image.ssim(
            tf.cast(y_t, tf.float32),
            tf.cast(y_p, tf.float32),
            max_val=1.0
        ).numpy()

        ssim_list.append(vals)

    ssim_vals = np.concatenate(ssim_list)
    return float(np.mean(ssim_vals)), float(np.std(ssim_vals))
